## ÉTAPE 8 — Clustering K-Means + ACP colorié
Objectif : regrouper les pays par profil agricole


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


## 1. PRÉPARER LES DONNÉES (même base que l'ACP)


In [ ]:
df = pd.read_csv("JEU_DE_DONNEE.csv")

cultures = [
    'cereals_total', 'roots_and_tubers_total', 'vegetables_melons_total',
    'fruit_excl_melons_total', 'oilcrops_primary', 'pulses_total',
    'wheat', 'maize', 'rice_paddy', 'sugar_cane'
]

df_pays = df[
    (df["element"] == "Production Quantity") &
    (~df["country_or_area"].str.contains(r"\+", na=False))
].dropna(subset=["value"])

tableau = (
    df_pays[df_pays["category"].isin(cultures)]
    .groupby(["country_or_area", "category"])["value"]
    .sum()
    .reset_index()
    .pivot_table(index="country_or_area", columns="category", values="value", aggfunc="sum")
    .fillna(0)
)

# Standardisation + ACP
scaler = StandardScaler()
X = scaler.fit_transform(tableau)

pca = PCA(n_components=2)
coords = pca.fit_transform(X)
var_pc1 = pca.explained_variance_ratio_[0] * 100
var_pc2 = pca.explained_variance_ratio_[1] * 100

print(f"Tableau : {tableau.shape[0]} pays × {tableau.shape[1]} cultures")
print(f"Variance conservée : {var_pc1 + var_pc2:.1f}%\n")


## 2. CHOISIR LE MEILLEUR NOMBRE DE CLUSTERS


## Le score de silhouette mesure la qualité des clusters
Plus il est proche de 1, mieux les groupes sont séparés


In [ ]:
print("--- Score silhouette selon le nombre de clusters ---")
scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)
    print(f"  k={k} → {scores[k]:.3f}")

meilleur_k = max(scores, key=scores.get)
print(f"\n→ Meilleur k : {meilleur_k} clusters (silhouette = {scores[meilleur_k]:.3f})")


## 3. GRAPHIQUE — COURBE DU SCORE SILHOUETTE


In [ ]:
fig0, ax0 = plt.subplots(figsize=(8, 4))
ax0.plot(list(scores.keys()), list(scores.values()),
         marker="o", linewidth=2.5, color="#2563EB", markersize=8)
ax0.axvline(x=4, color="orange", linestyle="--", linewidth=1.5,
            label="k=4 choisi (bon équilibre)")
ax0.set_title("Score silhouette selon le nombre de clusters", fontsize=13, fontweight="bold")
ax0.set_xlabel("Nombre de clusters (k)")
ax0.set_ylabel("Score silhouette (plus proche de 1 = mieux)")
ax0.set_xticks(range(2, 9))
ax0.legend()
ax0.grid(alpha=0.4)
plt.tight_layout()
plt.savefig("silhouette_scores.png", dpi=150, bbox_inches="tight")
print("\nGraphique silhouette sauvegardé ✓")
plt.show()


## 4. APPLIQUER K-MEANS AVEC k=4


## On choisit k=4 : meilleur équilibre entre
score silhouette (k=2 est trop grossier)
et interprétabilité (groupes distincts)


In [ ]:
k_choisi = 4
km = KMeans(n_clusters=k_choisi, random_state=42, n_init=10)
labels = km.fit_predict(X)

# Regrouper les résultats dans un DataFrame
df_res = pd.DataFrame({
    "pays"    : tableau.index,
    "PC1"     : coords[:, 0],
    "PC2"     : coords[:, 1],
    "cluster" : labels
})

# Nommer les clusters selon leur composition
noms_clusters = {
    0: "Petits producteurs (213 pays)",
    1: "Grands producteurs émergents",
    2: "Géant asiatique (Chine)",
    3: "Leader mondial (USA)"
}

couleurs_clusters = {
    0: "#93C5FD",  # bleu clair  — petits producteurs
    1: "#16A34A",  # vert        — émergents
    2: "#DC2626",  # rouge       — Chine
    3: "#D97706",  # orange      — USA
}

print("\n--- Composition des clusters ---")
for c in sorted(df_res["cluster"].unique()):
    pays = df_res[df_res["cluster"] == c]["pays"].tolist()
    print(f"\nCluster {c} — {noms_clusters[c]} :")
    print(f"  {pays[:10]}{'...' if len(pays) > 10 else ''}")


## 5. GRAPHIQUE ACP COLORIÉ PAR CLUSTER


In [ ]:
pays_a_annoter = {
    "China"                    : "Chine",
    "United States of America" : "USA",
    "India"                    : "Inde",
    "Brazil"                   : "Brésil",
    "France"                   : "France",
    "Russian Federation"       : "Russie",
    "Australia"                : "Australie",
    "Argentina"                : "Argentine",
    "Germany"                  : "Allemagne",
    "Indonesia"                : "Indonésie",
    "Pakistan"                 : "Pakistan",
    "Canada"                   : "Canada",
}

fig, ax = plt.subplots(figsize=(14, 9))

# Tracer chaque cluster séparément
for c in sorted(df_res["cluster"].unique()):
    groupe = df_res[df_res["cluster"] == c]
    ax.scatter(
        groupe["PC1"], groupe["PC2"],
        color=couleurs_clusters[c],
        s=60 if c == 0 else 120,
        alpha=0.7 if c == 0 else 1.0,
        zorder=3 if c > 0 else 2,
        label=f"Cluster {c} — {noms_clusters[c]}"
    )

# Annoter les pays importants
for _, row in df_res.iterrows():
    if row["pays"] in pays_a_annoter:
        ax.annotate(
            pays_a_annoter[row["pays"]],
            xy=(row["PC1"], row["PC2"]),
            xytext=(row["PC1"] + 0.2, row["PC2"] + 0.2),
            fontsize=9, fontweight="bold",
            color=couleurs_clusters[row["cluster"]]
        )

# Lignes centrales
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.4)
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.4)

ax.set_title(
    "ACP + Clustering K-Means — Profils agricoles (1961–2007)",
    fontsize=15, fontweight="bold", pad=15
)
ax.set_xlabel(f"PC1 — {var_pc1:.1f}% de variance", fontsize=12)
ax.set_ylabel(f"PC2 — {var_pc2:.1f}% de variance", fontsize=12)
ax.legend(fontsize=10, loc="upper right", framealpha=0.9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("acp_kmeans.png", dpi=150, bbox_inches="tight")
print("\nGraphique ACP K-Means sauvegardé ✓")
plt.show()


## 6. PROFIL MOYEN DE CHAQUE CLUSTER


## Quelle culture caractérise chaque groupe ?


In [ ]:
tableau_avec_cluster = tableau.copy()
tableau_avec_cluster["cluster"] = labels

profil = (
    tableau_avec_cluster
    .groupby("cluster")[cultures]
    .mean()
    .div(1_000_000)  # en millions de tonnes
)

print("\n--- Production moyenne par cluster (millions de tonnes) ---")
print(profil.round(1).to_string())


## 7. GRAPHIQUE — PROFIL MOYEN EN BARRES GROUPÉES


In [ ]:
fig2, ax2 = plt.subplots(figsize=(14, 6))

x = np.arange(len(cultures))
largeur = 0.2

for i, c in enumerate(sorted(profil.index)):
    ax2.bar(
        x + i * largeur,
        profil.loc[c],
        largeur,
        label=f"Cluster {c}",
        color=couleurs_clusters[c],
        alpha=0.9
    )

ax2.set_title(
    "Profil agricole moyen par cluster (millions de tonnes)",
    fontsize=14, fontweight="bold"
)
ax2.set_xticks(x + largeur * 1.5)
ax2.set_xticklabels(cultures, rotation=30, ha="right", fontsize=10)
ax2.set_ylabel("Production moyenne (Mt)")
ax2.legend(fontsize=10)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("profil_clusters.png", dpi=150, bbox_inches="tight")
print("Graphique profils sauvegardé ✓")
plt.show()


## 8. RÉSUMÉ FINAL


In [ ]:
print("\n=== RÉSUMÉ DU CLUSTERING ===")
for c in sorted(df_res["cluster"].unique()):
    nb = len(df_res[df_res["cluster"] == c])
    print(f"Cluster {c} ({nb} pays) : {noms_clusters[c]}")

print(f"\nScore silhouette final (k=4) : {silhouette_score(X, labels):.3f}")
print("→ Proche de 1 = clusters bien séparés ✓")
